<a href="https://colab.research.google.com/github/djwillichile/geoia-bloom-huasco/blob/main/notebooks/dinamica_bloom_huasco_ndvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

# Dinámica de la floración (proxy de Desierto Florido) en la cuenca del Huasco usando MODIS MOD13Q1 (NDVI de 16 días)
---

</div>





## Propósito del notebook
Este notebook desarrolla un flujo de trabajo reproducible en **Google Earth Engine (API de Python)** para analizar la dinámica espacio-temporal de la vegetación en la cuenca del Huasco, utilizando el **NDVI** como un proxy del bloom asociada al fenómeno conocido como **Desierto Florido**.

## Objetivos específicos
1. Construir una **climatología de NDVI** en intervalos de 16 días a partir de la colección **MOD13Q1**.
2. Calcular **anomalías de NDVI** respecto de la climatología estacional esperada.
3. Identificar píxeles con condiciones compatibles con floración mediante la regla `NDVI > 0.20` y `NDVI_ANOM > 0.02`.
4. Generar una **serie temporal** con métricas resumidas para toda la cuenca.
5. Explorar los periodos y eventos más intensos de floración mediante composiciones de **NDVI máximo**.

## Idea metodológica
La lógica del análisis es comparar el valor observado de NDVI en cada compuesto MODIS con el valor medio esperado para ese mismo momento del año. De este modo, una anomalía positiva sugiere una respuesta de la vegetación superior a la condición normal de la temporada. Cuando esa respuesta supera, además, un umbral mínimo de verdor, se interpreta como una señal potencialmente asociada a episodios de floración.

## Salidas esperadas
- Serie temporal de área de floración potencial  (km²)
- Intensidad de floración (NDVI medio dentro de la floración)
- Media y desviación estándar del NDVI en toda la cuenca
- Anomalía media de la cuenca
- Gráficos de evolución temporal
- Composiciones espaciales de NDVI máximo para los eventos principales


## 0) Instalación e inicialización de Earth Engine

En esta sección se instalan los paquetes necesarios y se inicializa la sesión de Earth Engine.

> La primera vez que ejecutes el notebook tendrás que autenticarte.


In [1]:
# Instalar paquetes solo si no están instalados
import importlib.util
import os

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    if importlib.util.find_spec(package) is None:
        print(f"Instalando {pip_name}...")
        os.system(f"pip -q install {pip_name}")
    else:
        print(f"✅ {pip_name} ya está instalado")

install_if_missing("geemap")
install_if_missing("ee", "earthengine-api")

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# Inicialización de Earth Engine
from google.colab import userdata
# projectGEE = 'my-project'
projectGEE = userdata.get('projectGEE')

try:
    ee.Initialize(project=projectGEE)
    print("✅ Google Earth Engine inicializado")
except Exception as e:
    print("Iniciando autenticación...")
    ee.Authenticate()
    ee.Initialize(project=projectGEE)
    print("✅ Google Earth Engine inicializado tras autenticación")

✅ geemap ya está instalado
✅ earthengine-api ya está instalado
✅ Google Earth Engine inicializado


## 1) Definición del área de estudio

En esta parte se construye el area de interes mediente lsa geometrías de trabajo para la **cuenca del rio Huasco**.

### Componentes espaciales usados
- **GAUL**: para ubicar la provincia del Huasco
- **HydroBASINS**: para delimitar la cuenca hidrográfica


> Aunque se visualiza un AOI más amplio, las métricas se calculan **dentro de la geometría de la cuenca**.



In [ ]:
# -----------------------------
# Geometrías
# -----------------------------

# Provincia
gaul = ee.FeatureCollection("FAO/GAUL/2015/level2")

huasco_prov = (gaul
  .filter(ee.Filter.eq('ADM0_NAME', 'Chile'))
  .filter(ee.Filter.eq('ADM2_NAME', 'Huasco'))
  .geometry()
)

outlet = ee.Geometry.Point([-71.17, -28.47])

basins = ee.FeatureCollection('WWF/HydroSHEDS/v1/Basins/hybas_6')
huasco_basin = ee.Feature(basins.filterBounds(outlet).first()).geometry()

# Area de interés AOI (box con buffer)
AOI = huasco_basin.union(huasco_prov).bounds().buffer(5000).bounds()

print("✅ Geometrias cargadas")


✅ Geometrias cargadas


In [ ]:
# =============================
# Previsuacización del area de interés
# =============================
import geemap

# Habilitar widgets ipyleaflet en Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Definir función para agregar capas de referencia al mapa
def add_reference_layers(Map, zoom=8):
    # Centrar mapa
    Map.centerObject(AOI, zoom)

    # Contornos vectoriales
    Map.addLayer(ee.Image().paint(AOI, 1, 2), {"palette": ["blue"]}, "AOI")
    Map.addLayer(ee.Image().paint(huasco_basin, 1, 1.8), {"palette": ["red"]}, "Huasco basin")
    Map.addLayer(ee.Image().paint(huasco_prov, 1, 1.5), {"palette": ["darkblue"]}, "Huasco province")

    # Punto de salida
    Map.addLayer(outlet, {"color": "black"}, "Outlet")

# --- visual context ---
# Elevation
elev = ee.Image("USGS/SRTMGL1_003").clip(AOI)

# Hillshade
hillshade = ee.Terrain.hillshade(elev)

# Paletas de color
elevVis = {
    "min": 0,
    "max": 5500,
    "palette": [
        "#0d4f8b", "#2e8b57", "#a6d96a", "#fee08b",
        "#fdae61", "#d73027", "#7f3b08", "#f5f5f5",
    ],
    "opacity": 0.85
}

rgb = elev.visualize(**elevVis)

shaded = ee.Image.rgb(
    rgb.select(0).multiply(hillshade.divide(255)),
    rgb.select(1).multiply(hillshade.divide(255)),
    rgb.select(2).multiply(hillshade.divide(255))
)

# Generar mapa base
Map = geemap.Map()
Map.add_basemap("HYBRID")

# Map.addLayer(elev, elevVis, "Elevation (m)")
# Map.addLayer(hillshade, {"min": 0, "max": 255, "opacity": 0.3}, "Hillshade")

Map.addLayer(shaded, {}, "Topography shaded")

# Capas vectoriales de contexto
add_reference_layers(Map)

print("✅ Mapa interactivo desplegado")

Map

✅ Mapa interactivo desplegado


Map(center=[-28.859256987602333, -70.63855948678793], controls=(WidgetControl(options=['position', 'transparen…

## 2) Parámetros y funciones auxiliares

En esta sección se definen los parámetros generales del análisis, el período de estudio y las funciones auxiliares necesarias para preparar la colección MODIS y organizar la climatología estacional.

### Parámetros principales

Se trabaja con la variable `NDVI` del producto MOD13Q1, utilizando dos umbrales de referencia:

- `var_thr = 0.20`: umbral mínimo de NDVI para identificar condiciones de vegetación activa.
- `anom_thr = 0.02`: umbral mínimo de anomalía positiva respecto de la climatología, usado para detectar incrementos anómalos compatibles con eventos de floración.

Además, se define un período de análisis acotado entre `2014-01-01` y `2020-03-01`, con el fin de mantener controlado el volumen de datos durante la generación del portafolio temporal.

### Escala y región de análisis

Las métricas temporales se calculan sobre la cuenca del Huasco (`REGION = huasco_basin`).  
La escala base para las reducciones espaciales se fija en `250 m`, coherente con la resolución espacial nominal del producto MOD13Q1.

### Funciones auxiliares

Se implementan tres funciones principales:

- `add_slot16(img)`: asigna a cada imagen un identificador temporal (`slot16`) según su posición dentro del año, agrupando observaciones en intervalos de 16 días.
- `make_name(slot)`: genera nombres estandarizados para las imágenes climatológicas, siguiendo la convención `MOD13Q1/CLIM_001`, `CLIM_017`, ..., `CLIM_353`.
- `preprocess(img)`: aplica el preprocesamiento básico de la imagen, incluyendo:
  - selección de la variable de interés (`NDVI`),
  - enmascaramiento por calidad con `SummaryQA ≤ 1`,
  - recorte espacial al área de interés (`AOI`),
  - reescalado de los valores mediante el factor `0.0001`.

### Criterios metodológicos clave

- Filtro de calidad QA: `SummaryQA ≤ 1` (pixeles con calidad *Good* y *Marginal*).
- Organización temporal: `slot16 = floor(DOY / 16)`.
- Convención de nombres para climatología: `MOD13Q1/CLIM_001`, `MOD13Q1/CLIM_017`, ..., `MOD13Q1/CLIM_353`.

In [ ]:
# -----------------------------
# Parámetros
# -----------------------------
variable = 'NDVI'
var_thr = 0.20
anom_thr = 0.02

# Ventana de tiempo
START_DATE = '2015-01-01'
END_DATE   = '2019-12-31'

# Parametros para uso de métricas
REGION = huasco_basin
REGION_TAG  = "huasco_basin_bloom"
DATASET_TAG = "MOD13Q1"

SCALE_TS = 250      # increase to 1000 if needed
TILE_SCALE_TS = 8

def add_slot16(img):
    doy = ee.Date(img.get('system:time_start')).getRelative('day', 'year')  # 0..364/365
    slot16 = ee.Number(doy).divide(16).floor()
    return img.set('slot16', slot16)

def make_name(slot):
    doy_start = ee.Number(slot).multiply(16).add(1)      # 1,17,33...
    doy_str = doy_start.format('%03d')                   # 001,017,033...
    return ee.String('MOD13Q1/CLIM_').cat(doy_str)

def preprocess(img):
    qa = img.select('SummaryQA')
    good = qa.lte(1)
    return (img.select(variable)
            .updateMask(good)
            .clip(AOI)
            .multiply(0.0001)
            .copyProperties(img, img.propertyNames()))

def clim_image_for_slot(modis_col, slot, variable):
    slot = ee.Number(slot)
    mean = (modis_col
            .filter(ee.Filter.eq('slot16', slot))
            .mean()
            .rename(variable))
    name = make_name(slot)
    return mean.set({
        'slot16': slot,
        'climatology': 1,
        'system:index': name,
        'system:id': name,
        'system:time_start': ee.Date.fromYMD(2000, 1, 1).advance(slot.multiply(16), 'day').millis()
    })

## 3) Carga de MOD13Q1 y construcción de la climatología de 16 días por `slot16`

En esta etapa se carga la colección `MODIS/061/MOD13Q1` para el período de análisis y se aplican las funciones de preprocesamiento definidas anteriormente. Luego, cada imagen se clasifica según su posición dentro del año mediante la variable `slot16`.

A partir de esta organización temporal, se construye una climatología estacional del NDVI como referencia fenológica intra-anual calculando una imagen promedio de NDVI para cada `slot16`. El resultado es una colección de 23 imágenes climatológicas, que representan el comportamiento medio esperado del NDVI en intervalos sucesivos de aproximadamente 16 días.

Esta climatología servirá como base para estimar anomalías y detectar incrementos anómalos de vegetación compatibles con eventos de floración.

In [ ]:
# -----------------------------
# Cargar  MODIS MOD13Q1
# -----------------------------
modis_raw = ee.ImageCollection('MODIS/061/MOD13Q1').filterDate(START_DATE, END_DATE)
modis = modis_raw.map(preprocess).map(add_slot16)

# -----------------------------
# Construir climatología
# (imagen promedio de cada slot16)
# -----------------------------
slot_list = ee.List.sequence(0, 22)

modis_clim = ee.ImageCollection.fromImages(
    slot_list.map(lambda s: clim_image_for_slot(modis, s, variable))
)

print("✅ Imágenes de la climatología:", modis_clim.size().getInfo())

✅ Imágenes de la climatología: 23


In [ ]:
# -----------------------------
# Visualización NDVI máximo climatológico
# -----------------------------
modis_clim_max = (
    modis_clim.max()
    .rename(f'{variable}_clim_max')
    .clip(huasco_basin)
    .reproject(crs='EPSG:4326', scale=250)
    .resample('bicubic')
)


# Parámetros de visualización
vis_ndvi = {
    'min': 0.0,
    'max': 0.9,
    'palette': [
        '#ffffff', '#ce7e45', '#df923d', '#f1b555', '#fcd163', '#99b718',
        '#74a901', '#66a000', '#529400', '#3e8601', '#207401', '#056201',
        '#004c00', '#023b01', '#012e01', '#011d01', '#011301'
    ]
}

ndvi_rgb = modis_clim_max.visualize(**vis_ndvi)

ndvi_shaded = ee.Image.rgb(
    ndvi_rgb.select(0).multiply(hillshade.divide(255)),
    ndvi_rgb.select(1).multiply(hillshade.divide(255)),
    ndvi_rgb.select(2).multiply(hillshade.divide(255))
)


Map = foliumap.Map()
Map.add_basemap("HYBRID")

# Agregar al mapa
Map.addLayer(shaded, {}, "Topography shaded")
Map.addLayer(ndvi_shaded, {}, "NDVI Máximo climatológico")
add_reference_layers(Map)

Map

NameError: name 'foliumap' is not defined

## 4) Incorporación de bandas derivadas

En esta etapa, a cada imagen preprocesada se le asocia su climatología de referencia según el valor de `slot16`. A partir de esta comparación se calcula una banda de anomalía, definida como la diferencia entre el NDVI observado y el NDVI medio esperado para ese intervalo temporal del año.

Las bandas derivadas se definen de la siguiente manera:

- `Anomaly = NDVI - CLIM(slot16)`
- `Bloom = (NDVI > 0.20) AND (Anomaly > 0.02)`

La banda `Anomaly` permite cuantificar cuánto se desvía la vegetación observada respecto de la condición estacional media. Por su parte, la banda binaria `Bloom` identifica los píxeles donde coinciden dos condiciones: un valor absoluto de NDVI superior al umbral definido y una anomalía positiva suficientemente marcada.

De esta forma, cada imagen queda enriquecida con información derivada que permite distinguir no solo la magnitud del verdor, sino también si ese verdor es inusualmente alto respecto de la condición esperada para la fecha. Esta base será utilizada posteriormente para resumir la dinámica temporal y espacial de la floración en la cuenca del Huasco.


In [ ]:
# -----------------------------
# Añadir bandas derivadas
# -----------------------------

# Definir funciones
def clim_for_img(img):
    s = ee.Number(img.get('slot16'))
    return ee.Image(modis_clim.filter(ee.Filter.eq('slot16', s)).first())

def add_anom_bloom(img):
    clim = clim_for_img(img)
    anom = img.subtract(clim).rename('Anomaly')
    bloom = (img.gt(var_thr)
             .And(anom.gt(anom_thr))
             .selfMask()
             .rename('Bloom'))

    out = (img.rename(variable)
           .addBands(anom)
           .addBands(bloom)
           .copyProperties(img, img.propertyNames())
           .set({
               'slot16': img.get('slot16'),
               'clim_id': clim.get('system:id'),
               variable + '_thr': var_thr,
               'anom_thr': anom_thr
           }))
    return out

# Añadir bandas
modis = modis.map(add_anom_bloom)

print("✅ Bandas:", modis.first().bandNames().getInfo())


In [ ]:
# mcd12 = (ee.ImageCollection("MODIS/061/MCD12Q1")
#          .filterDate(START_DATE, END_DATE)
#          .first()
#          .select("LC_Type1")
#          .clip(AOI))


# igbpLandCoverVis = {
#   "min": 1.0,
#   "max": 17.0,
#   "palette": [
#     '05450a', '086a10', '54a708', '78d203', '009900', 'c6b044', 'dcd159',
#     'dade48', 'fbff13', 'b6ff05', '27ff87', 'c24f44', 'a5a5a5', 'ff6d4c',
#     '69fff8', 'f9ffa4', '1c0dff'
#   ],
# };

# Map = foliumap.Map()
# Map.add_basemap("HYBRID")
# Map.addLayer(mcd12, igbpLandCoverVis, 'IGBP Land Cover');
# Map.addLayer(ndvi_shaded, {}, "NDVI Máximo climatológico")
# add_reference_layers(Map)

# Map

## 5) Métricas de serie temporal para la cuenca

Para cada imagen compuesta de 16 días se calculan las siguientes métricas:

- `bloom_area_km2`: suma de `pixelArea` donde `Bloom = 1`, expresada en km²
- `bloom_ndvi_mean`, `bloom_ndvi_sd`: media y desviación estándar del NDVI dentro de los píxeles clasificados como floración
- `basin_ndvi_mean`, `basin_ndvi_sd`: media y desviación estándar del NDVI en toda la cuenca
- `basin_anom_mean`: anomalía media del NDVI en toda la cuenca

In [ ]:
# Calcular métricas espaciales por fecha
def metrics_feature(img):
    date = ee.Date(img.get('system:time_start'))

    # Área solo donde hay bloom
    area_img = ee.Image.pixelArea().updateMask(img.select('Bloom')).rename('bloom_area')

    # NDVI solo en píxeles bloom
    bloom_var = img.select(variable).updateMask(img.select('Bloom')).rename('bloom_var')

    # Combinar bandas para resumir en una sola reducción
    combined = img.select([variable, 'Anomaly']).addBands([bloom_var, area_img])

    # Reducers combinados
    reducers = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.sum(),
        sharedInputs=True
    )

    stats = combined.reduceRegion(
        reducer=reducers,
        geometry=REGION,
        scale=SCALE_TS,
        bestEffort=True,
        tileScale=TILE_SCALE_TS,
        maxPixels=1e13
    )

    # Crear una fila por fecha
    return ee.Feature(None, {
        'date': date.format('YYYY-MM-dd'),
        'time_start': date.millis(),
        'slot16': img.get('slot16'),
        'bloom_area_km2': ee.Number(stats.get('bloom_area_sum')).divide(1e6),
        'bloom_' + variable + '_mean': stats.get('bloom_var_mean'),
        'bloom_' + variable + '_sd': stats.get('bloom_var_stdDev'),
        'basin_' + variable + '_mean': stats.get(variable + '_mean'),
        'basin_' + variable + '_sd': stats.get(variable + '_stdDev'),
        'basin_anom_mean': stats.get('Anomaly_mean')
    })

# Ordenar colección y construir la tabla temporal
modis_portfolio = modis.filterDate(START_DATE, END_DATE).sort('system:time_start')
bloom_ts_fc = ee.FeatureCollection(modis_portfolio.map(metrics_feature))

print("✅ Total de imágenes procesadas:", bloom_ts_fc.size().getInfo())

In [ ]:
# Ver las primeras 5 filas de la tabla generada
features = bloom_ts_fc.limit(20).getInfo()['features']

# Extraer solo las propiedades
data = [f['properties'] for f in features]

# Convertir a DataFrame
# import pandas as pd #(ya está importado)
df_preview = pd.DataFrame(data)

# Mostrar tabla
display(df_preview.head())

## 6) Opciones de exportación de la serie temporal

En esta sección se preparan los nombres de salida para exportar la tabla de métricas.

Se consideran dos alternativas:

### Opción A$^*$: Exportación a Google Drive
Permite exportar la tabla directamente desde Earth Engine sin usar `getInfo()`.

### Opción B: Conversión a `pandas.DataFrame`
Permite trabajar la serie temporal localmente en Python y guardarla como archivo CSV.


$^*$ Recomendado

In [ ]:
# -----------------------------
# Preparar nombres para la exportación
# -----------------------------

import re

# Normalizar strings
def slug(s):
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

start_year = pd.to_datetime(START_DATE).year
end_year   = pd.to_datetime(END_DATE).year

OUT_BASENAME = f"{slug(REGION_TAG)}_{slug(DATASET_TAG)}_{slug(variable)}_{start_year}_{end_year}"

# Nombres finales
OUT_DRIVE_DESC   = f"{OUT_BASENAME}_ts"
OUT_DRIVE_PREFIX = f"{OUT_BASENAME}_timeseries"
OUT_LOCAL_CSV    = f"{OUT_DRIVE_PREFIX}.csv"

print("✅ Output names ready:")
print("  - Drive description :", OUT_DRIVE_DESC)
print("  - Drive file prefix :", OUT_DRIVE_PREFIX)
print("  - Local CSV         :", OUT_LOCAL_CSV)

In [ ]:
# -----------------------------
# Option A: Exportar a Google Drive (server-side)
# -----------------------------

import ipywidgets as widgets
from IPython.display import display, clear_output

# Encabezado y botones
titulo = widgets.HTML(f"<b>Archivo:</b> {OUT_DRIVE_PREFIX}.csv")
btn_export = widgets.Button(description="Exportar a Drive", button_style='success')
btn_cancel = widgets.Button(description="Cancelar", button_style='danger')

# Panel de acciones
acciones = widgets.VBox([
    titulo,
    widgets.HBox([btn_export, btn_cancel])
])

# Área de salida para mensajes
out = widgets.Output()

display(acciones, out)

def on_export_clicked(b):
    acciones.close()  # Oculta solo el bloque de botones
    with out:
        clear_output()
        task = ee.batch.Export.table.toDrive(
            collection=bloom_ts_fc,
            description=OUT_DRIVE_DESC,
            fileNamePrefix=OUT_DRIVE_PREFIX,
            fileFormat='CSV'
        )
        task.start()
        print(f"🚀 Exportación iniciada: {OUT_DRIVE_PREFIX}.csv enviándose a Google Drive.")

def on_cancel_clicked(b):
    acciones.close()  # Oculta solo el bloque de botones
    with out:
        clear_output()
        print("❌ Exportación cancelada. No se envió ningún archivo a Google Drive.")

btn_export.on_click(on_export_clicked)
btn_cancel.on_click(on_cancel_clicked)

In [ ]:
# -----------------------------
# Opción B: convertir a pandas DataFrame (client-side)
# -----------------------------

# Variable global para reutilizar después
df = None

# Encabezado y botones
titulo = widgets.HTML(f"<b>Archivo local:</b> {OUT_LOCAL_CSV}")
btn_df = widgets.Button(description="Generar DataFrame", button_style='info')
btn_save = widgets.Button(description="Generar y guardar CSV", button_style='success')
btn_cancel = widgets.Button(description="Cancelar", button_style='danger')

acciones = widgets.VBox([
    titulo, widgets.HBox([btn_df, btn_save, btn_cancel])
])

out = widgets.Output()

display(acciones, out)

def build_dataframe():
    features = bloom_ts_fc.map(lambda f: f.setGeometry(None)).getInfo()['features']
    data = [f['properties'] for f in features]

    df_local = pd.DataFrame(data)

    num_cols = [
        'bloom_area_km2',
        f'bloom_{variable}_mean',
        f'bloom_{variable}_sd',
        f'basin_{variable}_mean',
        f'basin_{variable}_sd',
        'basin_anom_mean'
    ]

    for c in num_cols:
        if c in df_local.columns:
            df_local[c] = pd.to_numeric(df_local[c], errors='coerce')

    if 'time_start' in df_local.columns:
        df_local['date'] = pd.to_datetime(df_local['time_start'], unit='ms')
        df_local = df_local.sort_values('time_start').reset_index(drop=True)

    return df_local

def on_df_clicked(b):
    global df
    acciones.close()

    with out:
        clear_output()
        print("⏳ Generando DataFrame desde Earth Engine...")

        df = build_dataframe()

        print(f"✅ DataFrame generado correctamente.")
        print(f"📊 Total de registros: {len(df)}")
        print("🧠 El DataFrame quedó disponible en la variable: df")

        display(df.head())

def on_save_clicked(b):
    global df
    acciones.close()

    with out:
        clear_output()

        if os.path.exists(OUT_LOCAL_CSV):
            os.remove(OUT_LOCAL_CSV)
            print(f"🗑️ Archivo local previo eliminado: {OUT_LOCAL_CSV}")

        print("⏳ Generando DataFrame y guardando CSV...")

        df = build_dataframe()
        df.to_csv(OUT_LOCAL_CSV, index=False)

        print(f"✅ DataFrame generado y archivo guardado: {OUT_LOCAL_CSV}")
        print(f"📊 Total de registros: {len(df)}")
        print("🧠 El DataFrame quedó disponible en la variable: df")

        display(df.head())

def on_cancel_clicked(b):
    acciones.close()
    with out:
        clear_output()
        print("❌ Proceso cancelado. No se generó el DataFrame ni el archivo CSV.")

btn_df.on_click(on_df_clicked)
btn_save.on_click(on_save_clicked)
btn_cancel.on_click(on_cancel_clicked)

## 7) Detección preliminar de eventos de bloom a partir de la serie temporal

En esta sección se analiza la evolución temporal del área con condición de floración potencial (`bloom_area_km2`) en la cuenca del Huasco. El objetivo es identificar episodios relevantes de bloom a partir de la serie temporal, destacando tanto su magnitud como su duración aproximada.

Para ello, se compara la serie original con una versión suavizada, lo que permite reducir la variabilidad de corto plazo y resaltar patrones temporales de mayor escala. Sobre esta base se detectan peaks y se delimitan ventanas preliminares de evento, que luego servirán para resumir y comparar episodios de floración.

## 7) Detección de eventos mayores de bloom

En esta sección se identifican eventos relevantes de bloom a partir de la serie temporal de área con floración potencial (`bloom_area_km2`). Para ello, se aplica un suavizado temporal, se detectan peaks principales y luego se delimitan ventanas de evento mediante un umbral relativo respecto del peak.

La detección se realiza sobre la serie suavizada para reducir el efecto de fluctuaciones de corto plazo, mientras que la visualización final puede referirse a los máximos observados dentro de cada evento.

In [ ]:
# =====================================================
# Preparación de la serie temporal
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter, find_peaks

assert 'df' in globals(), "No se encontró 'df'. Ejecuta primero la sección donde generas el DataFrame."

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

x = df['date']
y = df['bloom_area_km2'].fillna(0).astype(float).to_numpy()

In [ ]:
# =====================================================
# Suavizado de la serie temporal
# =====================================================

# SG_WINDOW: número de puntos (impar). 11 puntos × 16 días ≈ 176 días;
#   suficiente para suavizar ruido sin eliminar eventos estacionales.
# SG_POLY: orden del polinomio. 3 preserva bien los picos sin sobre-ajustar.
SG_WINDOW = 11   # debe ser impar
SG_POLY = 3

assert len(y) >= SG_WINDOW, (
    f"Serie demasiado corta ({len(y)} puntos) para SG_WINDOW={SG_WINDOW}. "
    "Reduce SG_WINDOW o amplía el rango de fechas."
)

y_smooth = savgol_filter(y, SG_WINDOW, SG_POLY)
df['bloom_area_smooth'] = y_smooth

In [ ]:
# =====================================================
# Detección de peaks principales
# =====================================================

MIN_DISTANCE = 12   # mínimo 12 períodos × 16 días ≈ 192 días entre picos
# percentil 90 como referencia del nivel de bloom significativo;
# factor 0.5 exige que el pico sobresalga al menos la mitad de ese nivel.
PROMINENCE = np.nanpercentile(y_smooth, 90) * 0.5
MIN_PEAK_HEIGHT = np.nanpercentile(y_smooth, 80)   # percentil 80: descarta picos menores

peaks, props = find_peaks(
    y_smooth,
    distance=MIN_DISTANCE,
    prominence=PROMINENCE,
    height=MIN_PEAK_HEIGHT
)

print("✅ Peaks principales detectados:", len(peaks))

In [ ]:
# =====================================================
# Delimitación de eventos usando umbral relativo al peak
# =====================================================

EVENT_RELATIVE_THRESHOLD = 0.45   # 45% del pico: delimita los flancos del evento
MIN_DURATION = 4   # 4 períodos × 16 días ≈ 64 días mínimo de duración
MIN_EVENT_AREA = np.nanpercentile(y_smooth, 80)   # área mínima para considerar evento

events = []

for p in peaks:
    peak_value = y_smooth[p]
    threshold = peak_value * EVENT_RELATIVE_THRESHOLD

    # Buscar inicio
    s = p
    while s > 0 and y_smooth[s] > threshold:
        s -= 1

    # Buscar fin
    e = p
    while e < len(y_smooth) - 1 and y_smooth[e] > threshold:
        e += 1

    duration = e - s

    if duration >= MIN_DURATION and peak_value >= MIN_EVENT_AREA:
        events.append((s, p, e))

print("✅ Eventos preliminares detectados:", len(events))

In [ ]:
# =====================================================
# Fusión de eventos cercanos
# =====================================================

MERGE_GAP = 1   # 1 período × 16 días: fusiona eventos separados por ≤16 días

merged_events = []
for ev in events:
    if not merged_events:
        merged_events.append(ev)
    else:
        s_prev, p_prev, e_prev = merged_events[-1]
        s, p, e = ev

        if s - e_prev <= MERGE_GAP:
            if y_smooth[p] > y_smooth[p_prev]:
                merged_events[-1] = (s_prev, p, e)
            else:
                merged_events[-1] = (s_prev, p_prev, e)
        else:
            merged_events.append(ev)

events = merged_events

print("✅ Eventos fusionados:", len(events))

In [ ]:
# =====================================================
# Tabla resumen de eventos
# =====================================================

event_rows = []
for i, (s, p, e) in enumerate(events, start=1):
    p_raw = s + np.argmax(y[s:e+1])

    event_rows.append({
        "event_id": i,
        "start": x.iloc[s],
        "peak_smooth": x.iloc[p],
        "peak_observed": x.iloc[p_raw],
        "end": x.iloc[e],
        "duration_days": (x.iloc[e] - x.iloc[s]).days,
        "peak_bloom_area_km2_raw": y[p_raw],
        "peak_bloom_area_km2_smooth": y_smooth[p],
        "mean_bloom_area_km2_event": np.mean(y[s:e+1]),
        "sum_bloom_area_km2_event": np.sum(y[s:e+1])
    })

events_df = pd.DataFrame(event_rows).sort_values(
    "peak_bloom_area_km2_smooth", ascending=False
).reset_index(drop=True)

print("✅ Eventos mayores de bloom detectados:")
display(events_df)

In [ ]:
# =====================================================
# Visualización de eventos detectados
# =====================================================

fig, ax = plt.subplots(figsize=(14, 5.8))

# Serie observada y suavizada
ax.plot(x, y, color='gray', alpha=0.30, linewidth=1.0, label='Área bloom observada')
ax.plot(x, y_smooth, color='darkgreen', linewidth=2.4, label='Área bloom suavizada')

# Línea del promedio del área observada
y_mean = np.nanmean(y)
ax.axhline(
    y=y_mean,
    color='steelblue',
    linestyle='--',
    linewidth=1.6,
    alpha=0.9,
    label='Promedio del área bloom'
)

# Para no repetir etiquetas en la leyenda
peak_label_added = False
event_label_added = False

for i, (s, p, e) in enumerate(events, start=1):
    p_raw = s + np.argmax(y[s:e+1])

    # Sombreado del evento
    ax.axvspan(
        x.iloc[s], x.iloc[e],
        color='gold',
        alpha=0.18,
        label='Ventana del evento' if not event_label_added else None
    )
    event_label_added = True

    # Peak observado dentro del evento
    ax.scatter(
        x.iloc[p_raw],
        y[p_raw],
        color='darkred',
        s=50,
        zorder=5,
        label='Peak observado del evento' if not peak_label_added else None
    )
    peak_label_added = True

    # Etiqueta del año
    ax.text(
        x.iloc[p_raw],
        y[p_raw] * 1.03,
        x.iloc[p_raw].strftime('%b-%Y'),
        ha='center',
        va='bottom',
        fontsize=9,
        color='darkred'
    )

ax.set_title('Detección de mayores eventos de bloom en la cuenca del Huasco', fontsize=13)
ax.set_xlabel('Fecha')
ax.set_ylabel('Área bloom (km²)')
ax.grid(True, alpha=0.3)

# Eje x cada 18 meses
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
# ax.set_xlim(x.min(), x.max())
ax.margins(x=0.01)
plt.xticks(rotation=45)

# Límite superior con margen
y_max = max(np.nanmax(y), np.nanmax(y_smooth))
ax.set_ylim(0, y_max * 1.18)

# Leyenda
ax.legend(frameon=False, ncol=1)

plt.tight_layout()
plt.show()

print("✅ Detección completada")

## 8) Caracterización de los eventos principales de bloom

En esta sección se enriquecen los eventos detectados previamente con métricas adicionales de intensidad y contexto, tanto a nivel del peak como de la ventana completa del evento. El objetivo es comparar los episodios de bloom no solo por su extensión superficial, sino también por su duración, intensidad de verdor y magnitud de la anomalía.

Para cada evento se extraen métricas asociadas al peak observado y se calculan estadísticos resumidos dentro de la ventana temporal del evento. Esto permite construir una tabla comparativa útil para identificar los eventos más relevantes del período analizado.

In [ ]:
# =====================================================
# Caracterización de eventos principales de bloom
# =====================================================

assert 'events' in globals(), "No se encontró 'events'. Ejecuta primero la detección de eventos."
assert 'df' in globals(), "No se encontró 'df'. Ejecuta primero la generación del DataFrame."

event_rows = []

for i, (s, p, e) in enumerate(events, start=1):
    # Peak observado dentro de la ventana del evento
    p_raw = s + np.argmax(y[s:e+1])

    # Subconjunto temporal del evento
    event_df = df.iloc[s:e+1].copy()

    # Fila correspondiente al peak observado
    peak_row = df.iloc[p_raw]

    event_rows.append({
        "event_id": i,
        "start": x.iloc[s],
        "peak_smooth": x.iloc[p],
        "peak_observed": x.iloc[p_raw],
        "end": x.iloc[e],
        "duration_days": (x.iloc[e] - x.iloc[s]).days,

        # Magnitud del evento
        "peak_bloom_area_km2_raw": y[p_raw],
        "peak_bloom_area_km2_smooth": y_smooth[p],
        "mean_bloom_area_km2_event": event_df["bloom_area_km2"].mean(),
        "sum_bloom_area_km2_event": event_df["bloom_area_km2"].sum(),

        # Intensidad en el peak observado
        "peak_bloom_ndvi_mean": peak_row.get("bloom_NDVI_mean", np.nan),
        "peak_bloom_ndvi_sd": peak_row.get("bloom_NDVI_sd", np.nan),
        "peak_basin_ndvi_mean": peak_row.get("basin_NDVI_mean", np.nan),
        "peak_basin_ndvi_sd": peak_row.get("basin_NDVI_sd", np.nan),
        "peak_basin_anom_mean": peak_row.get("basin_anom_mean", np.nan),

        # Promedios durante toda la ventana del evento
        "mean_bloom_ndvi_event": event_df.get("bloom_NDVI_mean", pd.Series(dtype=float)).mean(),
        "mean_basin_ndvi_event": event_df.get("basin_NDVI_mean", pd.Series(dtype=float)).mean(),
        "mean_basin_anom_event": event_df.get("basin_anom_mean", pd.Series(dtype=float)).mean()
    })

events_char_df = pd.DataFrame(event_rows)

# Ordenar por área máxima observada
events_char_df = events_char_df.sort_values(
    "peak_bloom_area_km2_raw", ascending=False
).reset_index(drop=True)

print("✅ Tabla de caracterización de eventos generada:")
display(events_char_df)

## 9) Interpretación preliminar de los eventos detectados

La tabla de caracterización de eventos permite comparar los principales episodios de bloom en la cuenca del Huasco en términos de extensión, duración e intensidad.

De manera preliminar, se observa que los eventos más relevantes no necesariamente coinciden en todas sus dimensiones. En particular, un evento puede destacar por su gran superficie afectada, mientras otro puede presentar mayor intensidad de verdor dentro del bloom o una anomalía media más alta respecto de la climatología.

Esto sugiere que la dinámica del bloom no debe interpretarse únicamente a partir del área afectada, sino como un fenómeno multidimensional, donde la magnitud espacial, la persistencia temporal y la intensidad de la respuesta vegetacional pueden variar entre eventos.

A partir de esta caracterización, el siguiente paso consiste en comparar espacialmente la recurrencia e intensidad del bloom, con el fin de identificar sectores de la cuenca donde el fenómeno se expresa con mayor frecuencia o intensidad.

In [ ]:
# =====================================================
# Resumen automático de los eventos más relevantes
# =====================================================
# =====================================================
# Resumen interpretativo de eventos clave (sin índice visible)
# =====================================================

assert 'events_char_df' in globals(), "No se encontró 'events_char_df'. Ejecuta primero la caracterización de eventos."

summary_df = events_char_df.copy()

# Asegurar formato de fechas
for c in ["start", "peak_smooth", "peak_observed", "end"]:
    if c in summary_df.columns:
        summary_df[c] = pd.to_datetime(summary_df[c])

# Función para seleccionar y formatear una fila
def format_event_row(row):
    out = pd.DataFrame([{
        "Evento": int(row["event_id"]),
        "Inicio": row["start"].strftime("%Y-%m-%d"),
        "Peak observado": row["peak_observed"].strftime("%Y-%m-%d"),
        "Fin": row["end"].strftime("%Y-%m-%d"),
        "Duración (días)": int(row["duration_days"]),
        "Área máxima bloom (km²)": round(float(row["peak_bloom_area_km2_raw"]), 2),
        "NDVI medio en bloom": round(float(row["peak_bloom_ndvi_mean"]), 3) if pd.notna(row["peak_bloom_ndvi_mean"]) else np.nan,
        "Anomalía media cuenca": round(float(row["peak_basin_anom_mean"]), 3) if pd.notna(row["peak_basin_anom_mean"]) else np.nan
    }])
    return out

# Función para mostrar sin índice
def display_no_index(df_show):
    display(df_show.style.hide(axis="index"))

# Evento más extenso
ev_max_area = summary_df.loc[summary_df["peak_bloom_area_km2_raw"].idxmax()]
print("✅ Evento más extenso:")
display_no_index(format_event_row(ev_max_area))
print("\n")

# Evento más duradero
ev_max_duration = summary_df.loc[summary_df["duration_days"].idxmax()]
print("✅ Evento más duradero:")
display_no_index(format_event_row(ev_max_duration))
print("\n")

# Evento con mayor intensidad de NDVI dentro del bloom
ev_max_ndvi = summary_df.loc[summary_df["peak_bloom_ndvi_mean"].idxmax()]
print("✅ Evento con mayor intensidad de NDVI dentro del bloom:")
display_no_index(format_event_row(ev_max_ndvi))
print("\n")

# Evento con mayor anomalía media en la cuenca
ev_max_anom = summary_df.loc[summary_df["peak_basin_anom_mean"].idxmax()]
print("✅ Evento con mayor anomalía media en la cuenca:")
display_no_index(format_event_row(ev_max_anom))
print("\n")

# Comparación simple: eventos grandes vs anómalos
same_area_anom = int(ev_max_area["event_id"]) == int(ev_max_anom["event_id"])

print("✅ Comparación entre extensión y anomalía")
if same_area_anom:
    print("El evento más extenso coincide con el evento de mayor anomalía media en la cuenca.")
else:
    print("El evento más extenso no coincide con el evento de mayor anomalía media en la cuenca.")
    print(f"- Evento más extenso: {int(ev_max_area['event_id'])}")
    print(f"- Evento más anómalo: {int(ev_max_anom['event_id'])}")

## 10) Frecuencia espacial del bloom

En esta sección se analiza la frecuencia espacial del bloom en la cuenca del Huasco, entendida como el número de veces que cada píxel fue clasificado con condición de floración potencial (`Bloom`) durante el período de estudio.

Esta visualización permite identificar sectores donde el fenómeno aparece de manera más recurrente, distinguiéndolos de aquellos donde el bloom ocurre solo de forma ocasional. En este sentido, la frecuencia de bloom constituye una medida de persistencia espacial del fenómeno dentro de la cuenca.

A diferencia de las métricas temporales anteriores, aquí el interés se centra en la distribución espacial acumulada del bloom, lo que permite reconocer posibles núcleos de recurrencia y áreas con mayor expresión del fenómeno a lo largo del período analizado.

In [ ]:
# =====================================================
# Frecuencia espacial del bloom
# =====================================================

# Calcular frecuencia de bloom por píxel
modis_bloom_freq = (
    modis.select('Bloom')
    .map(lambda img: img.unmask(0))
    .sum()
    .rename('bloom_frequency')
    .clip(huasco_basin)
    .reproject(crs='EPSG:4326', scale=250)
    .resample('bicubic')
)

# Obtener valor máximo para ajustar la visualización
freq_max_dict = modis_bloom_freq.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=huasco_basin,
    scale=250,
    bestEffort=True,
    maxPixels=1e13
)

freq_max = freq_max_dict.get('bloom_frequency').getInfo()
print("✅ Frecuencia máxima de bloom:", freq_max)

In [ ]:
# Parámetros de visualización
vis_bloom_freq = {
    'min': 0,
    'max': freq_max,
    'palette': [
        '#00008f', '#007fff', '#7fff7f', '#ff7f00', '#8f0000'
    ]
}

# Convertir a RGB para sombreado
bloom_freq_rgb = modis_bloom_freq.visualize(**vis_bloom_freq)

bloom_freq_shaded = ee.Image.rgb(
    bloom_freq_rgb.select(0).multiply(hillshade.divide(255)),
    bloom_freq_rgb.select(1).multiply(hillshade.divide(255)),
    bloom_freq_rgb.select(2).multiply(hillshade.divide(255))
).clip(huasco_basin)

# Crear mapa
Map = foliumap.Map()
Map.add_basemap("HYBRID")

# Agregar al mapa
Map.addLayer(shaded, {}, "Topography shaded")
Map.addLayer(bloom_freq_shaded, {}, "Frecuencia de bloom")
add_reference_layers(Map)

Map

In [ ]:
# =====================================================
# Construir lista de fechas correspondientes a eventos principales
# =====================================================

assert 'events' in globals(), "No se encontró 'events'. Ejecuta primero la detección de eventos."
assert 'df' in globals(), "No se encontró 'df'. Ejecuta primero la generación del DataFrame."

event_time_starts = []

for s, p, e in events:
    event_time_starts.extend(df.iloc[s:e+1]['time_start'].tolist())

# Eliminar duplicados y ordenar
event_time_starts = sorted(set(event_time_starts))

print("✅ Total de fechas incluidas en eventos principales:", len(event_time_starts))
print("Primeras fechas (time_start):", event_time_starts[:5])

In [ ]:
# =====================================================
# Filtrar la colección MODIS a fechas de eventos principales
# =====================================================

event_time_starts_ee = ee.List(event_time_starts)

modis_events = modis.filter(
    ee.Filter.inList('system:time_start', event_time_starts_ee)
)

print("✅ Imágenes en eventos principales:", modis_events.size().getInfo())

In [ ]:
# =====================================================
# Frecuencia espacial del bloom en eventos principales
# =====================================================
modis_bloom_freq_events = (
    modis_events.select('Bloom')
    .map(lambda img: img.unmask(0))
    .sum()
    .rename('bloom_frequency_events')
    .clip(huasco_basin)
    .reproject(crs='EPSG:4326', scale=250)
    .resample('bicubic')
)

# Valor máximo para visualización
freq_max_dict = modis_bloom_freq_events.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=huasco_basin,
    scale=250,
    bestEffort=True,
    maxPixels=1e13
)

freq_max_events = freq_max_dict.get('bloom_frequency_events').getInfo()
print("✅ Frecuencia máxima de bloom en eventos principales:", freq_max_events)

In [ ]:
# =====================================================
# Visualización de la frecuencia de bloom en eventos principales
# =====================================================

# Parámetros de visualización
vis_bloom_freq = {
    'min': 0,
    'max': freq_max_events,
    'palette': [
        '#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c'
    ]
}

# Convertir a RGB para sombreado
bloom_freq_rgb = modis_bloom_freq_events.visualize(**vis_bloom_freq)

bloom_freq_shaded = ee.Image.rgb(
    bloom_freq_rgb.select(0).multiply(hillshade.divide(255)),
    bloom_freq_rgb.select(1).multiply(hillshade.divide(255)),
    bloom_freq_rgb.select(2).multiply(hillshade.divide(255))
).clip(huasco_basin)

# Crear mapa
Map = foliumap.Map()
Map.add_basemap("HYBRID")

# Agregar capa temática
Map.addLayer(shaded, {}, "Topography shaded")
Map.addLayer(bloom_freq_shaded, {}, "Frecuencia de bloom en eventos principales")

# Agregar referencias
add_reference_layers(Map)

Map

In [ ]:
# =====================================================
# Cartografía estática con matplotlib:
# Frecuencia espacial del bloom en eventos principales
# =====================================================
# =====================================================
# Cartografía estática tipo lámina:
# Frecuencia espacial del bloom en eventos principales
# =====================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec
import geemap
import ee

# -----------------------------
# Verificaciones mínimas
# -----------------------------
assert 'modis_bloom_freq_events' in globals(), "No se encontró 'modis_bloom_freq_events'."
assert 'freq_max_events' in globals(), "No se encontró 'freq_max_events'."
assert 'huasco_basin' in globals(), "No se encontró 'huasco_basin'."
assert 'huasco_prov' in globals(), "No se encontró 'huasco_prov'."
assert 'AOI' in globals(), "No se encontró 'AOI'."
assert 'outlet' in globals(), "No se encontró 'outlet'."

# -----------------------------
# Funciones auxiliares
# -----------------------------
def region_extent(geom):
    coords = geom.bounds().coordinates().getInfo()[0]
    xs = [pt[0] for pt in coords]
    ys = [pt[1] for pt in coords]
    return [min(xs), max(xs), min(ys), max(ys)]

def ee_image_to_array(img, region, scale=250):
    arr = geemap.ee_to_numpy(img, region=region, scale=scale)
    if arr is None:
        raise ValueError("No se pudo extraer el array desde Earth Engine.")
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    return np.array(arr)

def add_north_arrow(ax, x=0.93, y=0.92, size=0.08):
    ax.annotate(
        'N',
        xy=(x, y),
        xytext=(x, y - size),
        xycoords='axes fraction',
        textcoords='axes fraction',
        arrowprops=dict(facecolor='black', width=3, headwidth=12),
        ha='center',
        va='center',
        fontsize=14,
        fontweight='bold'
    )

def add_scale_bar(ax, extent, length_km=20, y_frac=0.06, x_frac=0.08, text_offset_frac=0.012):
    xmin, xmax, ymin, ymax = extent
    mean_lat = (ymin + ymax) / 2
    km_per_degree_lon = 111.32 * np.cos(np.deg2rad(mean_lat))
    scale_deg = length_km / km_per_degree_lon

    x0 = xmin + (xmax - xmin) * x_frac
    y0 = ymin + (ymax - ymin) * y_frac

    ax.plot([x0, x0 + scale_deg], [y0, y0], color='black', linewidth=4, solid_capstyle='butt')
    ax.plot([x0, x0], [y0 - (ymax - ymin) * 0.005, y0 + (ymax - ymin) * 0.005], color='black', linewidth=2)
    ax.plot([x0 + scale_deg, x0 + scale_deg], [y0 - (ymax - ymin) * 0.005, y0 + (ymax - ymin) * 0.005], color='black', linewidth=2)
    ax.text(x0 + scale_deg / 2, y0 + (ymax - ymin) * text_offset_frac, f'{length_km} km',
            ha='center', va='bottom', fontsize=9, color='black')

# -----------------------------
# Parámetros cartográficos
# -----------------------------
palette = ['#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c']
cmap = LinearSegmentedColormap.from_list("bloom_freq", palette, N=256)

# Extensiones
main_extent = region_extent(huasco_basin)
inset_extent = region_extent(huasco_prov)

main_region = huasco_basin.bounds()
inset_region = huasco_prov.bounds()

# -----------------------------
# Raster principal
# -----------------------------
band_name = modis_bloom_freq_events.bandNames().getInfo()[0]
freq_arr = ee_image_to_array(modis_bloom_freq_events.select([band_name]), region=main_region, scale=250)
freq_arr = np.ma.masked_invalid(freq_arr)
freq_arr = np.ma.masked_where(freq_arr <= 0, freq_arr)

# -----------------------------
# Capas rasterizadas: mapa principal
# -----------------------------
basin_outline = ee.Image().byte().paint(huasco_basin, 1, 2)
aoi_outline = ee.Image().byte().paint(AOI, 1, 2)
prov_outline_main = ee.Image().byte().paint(huasco_prov, 1, 1)
outlet_img = ee.Image().byte().paint(outlet, 1, 4)

basin_arr = ee_image_to_array(basin_outline, region=main_region, scale=250)
aoi_arr = ee_image_to_array(aoi_outline, region=main_region, scale=250)
prov_arr_main = ee_image_to_array(prov_outline_main, region=main_region, scale=250)
outlet_arr = ee_image_to_array(outlet_img, region=main_region, scale=250)

# -----------------------------
# Capas rasterizadas: mapa de ubicación
# -----------------------------
prov_fill = ee.Image.constant(1).clip(huasco_prov).selfMask()
basin_fill = ee.Image.constant(1).clip(huasco_basin).selfMask()
prov_outline_inset = ee.Image().byte().paint(huasco_prov, 1, 1)
basin_outline_inset = ee.Image().byte().paint(huasco_basin, 1, 2)

prov_fill_arr = ee_image_to_array(prov_fill, region=inset_region, scale=500)
basin_fill_arr = ee_image_to_array(basin_fill, region=inset_region, scale=500)
prov_outline_arr = ee_image_to_array(prov_outline_inset, region=inset_region, scale=500)
basin_outline_arr = ee_image_to_array(basin_outline_inset, region=inset_region, scale=500)

# -----------------------------
# Figura y composición
# -----------------------------
fig = plt.figure(figsize=(14, 9))
gs = GridSpec(
    nrows=2, ncols=2,
    height_ratios=[15, 1.6],
    width_ratios=[1.1, 3.1],
    hspace=0.02, wspace=0.03
)

ax_inset = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[0, 1])
ax_leg = fig.add_subplot(gs[1, :])

# -----------------------------
# Mapa de ubicación (izquierda)
# -----------------------------
# Provincia
ax_inset.imshow(
    np.ma.masked_invalid(prov_fill_arr),
    cmap=LinearSegmentedColormap.from_list("prov", ['#d9d9d9', '#d9d9d9']),
    extent=inset_extent,
    origin='upper'
)

# Cuenca destacada
ax_inset.imshow(
    np.ma.masked_invalid(basin_fill_arr),
    cmap=LinearSegmentedColormap.from_list("basin", ['#6b6b6b', '#6b6b6b']),
    extent=inset_extent,
    origin='upper',
    alpha=0.95
)

# Contornos
ax_inset.contour(prov_outline_arr, levels=[0.5], colors=['#8c8c8c'], linewidths=0.8, extent=inset_extent)
ax_inset.contour(basin_outline_arr, levels=[0.5], colors=['#1f1f1f'], linewidths=1.2, extent=inset_extent)

ax_inset.set_title('Ubicación', fontsize=11, pad=6)
ax_inset.set_xticks([])
ax_inset.set_yticks([])
for spine in ax_inset.spines.values():
    spine.set_linewidth(1.0)
    spine.set_color('#333333')

# -----------------------------
# Mapa principal (derecha)
# -----------------------------
im = ax_main.imshow(
    freq_arr,
    cmap=cmap,
    vmin=0,
    vmax=float(freq_max_events),
    extent=main_extent,
    origin='upper'
)

# Referencias espaciales
ax_main.contour(prov_arr_main, levels=[0.5], colors=['#8f8f8f'], linewidths=0.8, extent=main_extent)
ax_main.contour(aoi_arr, levels=[0.5], colors=['#3182bd'], linewidths=1.0, extent=main_extent)
ax_main.contour(basin_arr, levels=[0.5], colors=['#e31a1c'], linewidths=1.4, extent=main_extent)
ax_main.contour(outlet_arr, levels=[0.5], colors=['black'], linewidths=2.2, extent=main_extent)

ax_main.set_xticks([])
ax_main.set_yticks([])
for spine in ax_main.spines.values():
    spine.set_linewidth(1.0)
    spine.set_color('#333333')

add_north_arrow(ax_main, x=0.95, y=0.95, size=0.10)
add_scale_bar(ax_main, main_extent, length_km=20)

# -----------------------------
# Panel de leyenda inferior
# -----------------------------
ax_leg.axis('off')

# Título y subtítulo generales
fig.suptitle(
    'Frecuencia espacial del bloom en eventos principales',
    fontsize=17, fontweight='bold', y=0.975
)
fig.text(
    0.5, 0.945,
    'Cuenca del Huasco · Período de análisis: 2014–2020',
    ha='center', va='center', fontsize=11
)

# Barra de color horizontal
cbar = fig.colorbar(im, ax=ax_leg, orientation='horizontal', fraction=0.45, pad=0.2)
cbar.set_label('Frecuencia de bloom (número de ocurrencias)', fontsize=10)

# Leyenda de referencias vectoriales
legend_elements = [
    Line2D([0], [0], color='#e31a1c', lw=1.5, label='Cuenca del Huasco'),
    Line2D([0], [0], color='#3182bd', lw=1.2, label='AOI'),
    Line2D([0], [0], color='#8f8f8f', lw=1.0, label='Provincia del Huasco'),
    Line2D([0], [0], marker='o', color='black', lw=0, markersize=7, label='Outlet')
]

ax_leg.legend(
    handles=legend_elements,
    loc='center',
    ncol=4,
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, -0.15)
)

# Fuente
fig.text(
    0.01, 0.02,
    'Fuente: MODIS MOD13Q1, Google Earth Engine\nProcesamiento y elaboración: Guillermo Fuentes',
    ha='left', va='bottom', fontsize=9
)

fig.text(
    0.99, 0.02,
    f'Valor máximo observado: {float(freq_max_events):.0f}',
    ha='right', va='bottom', fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 0.94])
plt.show()

# Guardado opcional
# fig.savefig('cartografia_frecuencia_bloom_eventos_huasco_lamina.png', dpi=300, bbox_inches='tight')

In [ ]:
# =====================================================
# Curva original, curvatura, primera derivada y segunda derivada
# =====================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Asegurar orden temporal
df = df.sort_values('date').reset_index(drop=True)

# Eje temporal
x_dates = pd.to_datetime(df['date'])
x_days = (x_dates - x_dates.iloc[0]).dt.days.to_numpy()

# Serie original y suavizada
y_raw = df['bloom_area_km2'].fillna(0).astype(float).to_numpy()
y_smooth = df['bloom_area_smooth'].astype(float).to_numpy()

# Primera y segunda derivada respecto al tiempo
dy = np.gradient(y_smooth, x_days)
ddy = np.gradient(dy, x_days)

# Curvatura geométrica
curvature = np.abs(ddy) / np.power(1 + dy**2, 1.5)

# Guardar en el DataFrame
df['dy'] = dy
df['ddy'] = ddy
df['curvature'] = curvature

# -----------------------------
# Gráfico
# -----------------------------
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

# 1) Área original + suavizada
axes[0].plot(x_dates, y_raw, color='gray', alpha=0.35, linewidth=1.0, label='Área observada')
axes[0].plot(x_dates, y_smooth, color='darkgreen', linewidth=2.2, label='Área suavizada')
axes[0].set_ylabel('Área bloom\n(km²)')
axes[0].set_title('Serie temporal del bloom y sus derivadas')
axes[0].grid(True, alpha=0.3)
axes[0].legend(frameon=False)

# 2) Primera derivada
axes[1].plot(x_dates, dy, linewidth=1.8)
axes[1].axhline(0, color='black', linewidth=0.8, alpha=0.7)
axes[1].set_ylabel("1ª derivada\n(dA/dt)")
axes[1].grid(True, alpha=0.3)

# 3) Segunda derivada
axes[2].plot(x_dates, ddy, linewidth=1.8)
axes[2].axhline(0, color='black', linewidth=0.8, alpha=0.7)
axes[2].set_ylabel("2ª derivada\n(d²A/dt²)")
axes[2].grid(True, alpha=0.3)

# 4) Curvatura
axes[3].plot(x_dates, curvature, linewidth=1.8)
axes[3].set_ylabel("Curvatura")
axes[3].set_xlabel("Fecha")
axes[3].grid(True, alpha=0.3)

# Formato del eje x
axes[3].xaxis.set_major_locator(mdates.YearLocator())
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Punto de parada opcional ───────────────────────────────────────────────
# Establece CONTINUE_ANALYSIS = True para ejecutar el flujo alternativo
# de análisis que viene a continuación (composites por evento).
CONTINUE_ANALYSIS = False

if not CONTINUE_ANALYSIS:
    print("ℹ️  Ejecución detenida antes del flujo alternativo.")
    print("   Cambia CONTINUE_ANALYSIS = True para continuar.")
    raise SystemExit(0)

## 7) Visualización de la serie temporal

En esta sección se generan gráficos básicos con `matplotlib` para explorar la evolución temporal de la señal de floración.

### Gráficos considerados
1. **Extensión potencial del bloom** (km²)
2. **Intensidad de la vegetación en zonas Bloom** versus NDVI medio de la cuenca
3. **Anomalía media de la cuenca** y variabilidad interna del NDVI

We plot:
1) Bloom extent (km²)
2) Bloom intensity (NDVI mean within bloom) vs basin NDVI mean
3) Basin anomaly mean and basin NDVI SD


In [ ]:
# =====================================================
# Global plotting settings & helper functions
# =====================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Convert and sort dates
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# --- Style knobs (feel free to tweak) ---
BLUE = "#1f77b4"          # main line
GREEN = "#014325"
BAND_ALPHA = 0.38         # shaded band opacity
LINE_W = .8
GRID_ALPHA = 0.5
FIGSIZE = (12.5, 3.4)

# Smoothing (optional): 5 steps ~ 80 days if MOD13Q1 16d
SMOOTH_WINDOW = 5

def rolling_mean(y, w):
    return pd.Series(y).rolling(w, center=True, min_periods=1).mean().to_numpy()

def rolling_std(y, w):
    return pd.Series(y).rolling(w, center=True, min_periods=1).std().to_numpy()

def plot_mean_with_band(
    x, y, y_sd=None,
    title="", ylab="", xlab="Date",
    smooth_window=None,
    show_legend=False,
    band_label="±1 SD",
    mean_label="Mean",
    clip_min=None
):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # Clean axes like your example
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=GRID_ALPHA, linewidth=0.8)

    # Band (± sd)
    if y_sd is not None:
        lower = y - y_sd
        upper = y + y_sd

        if clip_min is not None:
            lower = np.clip(lower, clip_min, None)
            upper = np.clip(upper, clip_min, None)

        ax.fill_between(
            x, lower, upper,
            color=BLUE, alpha=BAND_ALPHA, linewidth=0,
            label=band_label
        )

    # Smooth line (optional)
    if smooth_window is not None and smooth_window >= 3:
        y_s = rolling_mean(y, smooth_window)
        ax.plot(x, y_s, color=BLUE, linewidth=1.8, alpha=0.95,
                label=f"Rolling mean ({smooth_window})")
    # Mean line (changed to green points)
    ax.plot(x, y, color=GREEN, marker='o', markersize=3, linestyle='None', label=mean_label)

    # Date formatting (nice)
    ax.xaxis.set_major_locator(mdates.YearLocator())      # tick cada año
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1,7]))  # opcional: semestral
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))

    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(x.min(), x.max())

    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)

    if show_legend:
        ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()
    plt.show()

print("✅ Plot environment ready")

In [ ]:
# ------------------------------
# 1) Bloom extent (km²) + band (proxy temporal)
# ------------------------------
x = df["date"]
y_area = df["bloom_area_km2"].astype(float).to_numpy()

# Para área bloom no tienes SD espacial; usamos banda proxy temporal para estética tipo paper
y_area_sd_proxy = rolling_std(y_area, SMOOTH_WINDOW)

plot_mean_with_band(
    x=x,
    y=y_area,
    y_sd=y_area_sd_proxy,
    title="Bloom extent (km²) — Huasco Basin (band = rolling SD proxy)",
    ylab="Bloom area (km²)",
    smooth_window=SMOOTH_WINDOW,
    show_legend=True,
    band_label="±1 SD (rolling proxy)",
    mean_label="Bloom area",
    clip_min=0
)


In [ ]:
# ------------------------------
# 2) Bloom intensity vs basin NDVI mean
#    (líneas; y banda para basin si hay SD)
# ------------------------------
fig, ax = plt.subplots(figsize=(12.5, 3.6))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=GRID_ALPHA, linewidth=0.8)

y_bloom_mean = df[f"bloom_{variable}_mean"].astype(float).to_numpy()
y_basin_mean = df[f"basin_{variable}_mean"].astype(float).to_numpy()

# Banda SD espacial de la cuenca (si existe)
sd_col = f"basin_{variable}_sd"
if sd_col in df.columns:
    y_basin_sd = df[sd_col].astype(float).to_numpy()
    ax.fill_between(x, y_basin_mean - y_basin_sd, y_basin_mean + y_basin_sd,
                    color=BLUE, alpha=BAND_ALPHA, linewidth=0,
                    label=f"Basin {variable} ±1 SD")

# Líneas (promedios)
ax.plot(x, y_bloom_mean, color=BLUE, linewidth=LINE_W, label=f"Bloom {variable} mean")
ax.plot(x, y_basin_mean, color=BLUE, linewidth=LINE_W, alpha=0.55, label=f"Basin {variable} mean")

# Suavizados (opcional, para look tipo paper)
# ax.plot(x, rolling_mean(y_bloom_mean, SMOOTH_WINDOW), color=BLUE, linewidth=2.6, alpha=0.95,
#         label=f"Bloom {variable} rolling mean ({SMOOTH_WINDOW})")
# ax.plot(x, rolling_mean(y_basin_mean, SMOOTH_WINDOW), color=BLUE, linewidth=2.6, alpha=0.45,
#         label=f"Basin {variable} rolling mean ({SMOOTH_WINDOW})")

ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlim(x.min(), x.max())

ax.set_title(f"Bloom intensity vs Basin {variable} mean", fontsize=12, pad=10)
ax.set_xlabel("Date")
ax.set_ylabel(variable)
ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------
# 3) Basin anomaly mean + band (real if basin_anom_sd exists; else proxy)
# ------------------------------
y_anom = df["basin_anom_mean"].astype(float).to_numpy()

if "basin_anom_sd" in df.columns:
    y_anom_sd = df["basin_anom_sd"].astype(float).to_numpy()
    band_label = "±1 SD (spatial)"
    title = "Basin anomaly mean ± SD — Huasco Basin"
else:
    y_anom_sd = rolling_std(y_anom, SMOOTH_WINDOW)
    band_label = "±1 SD (rolling proxy)"
    title = "Basin anomaly mean — Huasco Basin (band = rolling SD proxy)"

plot_mean_with_band(
    x=x,
    y=y_anom,
    y_sd=y_anom_sd,
    title=title,
    ylab=f"{variable} anomaly",
    smooth_window=SMOOTH_WINDOW,
    show_legend=True,
    band_label=band_label,
    mean_label="Anomaly mean"
)


In [ ]:
# Convert dates
# df['date'] = pd.to_datetime(df['date'])

# plt.figure()
# plt.plot(df['date'], df['bloom_area_km2'])
# plt.title("Bloom extent (km²) — Huasco Basin")
# plt.xlabel("Date")
# plt.ylabel("km²")
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

# plt.figure()
# plt.plot(df['date'], df['bloom_' + variable + '_mean'], label="Bloom " + variable + " mean")
# plt.plot(df['date'], df['basin_' + variable + '_mean'], label="Basin " + variable + " mean")
# plt.title("Bloom intensity vs Basin NDVI mean")
# plt.xlabel("Date")
# plt.ylabel("NDVI")
# plt.legend()
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

# plt.figure()
# plt.plot(df['date'], df['basin_anom_mean'], label="Basin anomaly mean")
# plt.plot(df['date'], df['basin_' + variable + '_sd'], label="Basin " + variable + " SD")
# plt.title("Basin anomaly mean and NDVI heterogeneity (SD)")
# plt.xlabel("Date")
# plt.legend()
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ==============================
# Replace plotting cell
# ==============================

# Convert and sort dates
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# --- Style knobs (feel free to tweak) ---
BLUE = "#1f77b4"          # main line
BAND_ALPHA = 0.18         # shaded band opacity
LINE_W = 1.3
GRID_ALPHA = 0.25
FIGSIZE = (12.5, 3.4)

# Smoothing (optional): 5 steps ~ 80 days if MOD13Q1 16d
SMOOTH_WINDOW = 5

def rolling_mean(y, w):
    return pd.Series(y).rolling(w, center=True, min_periods=1).mean().to_numpy()

def rolling_std(y, w):
    return pd.Series(y).rolling(w, center=True, min_periods=1).std().to_numpy()

def plot_mean_with_band(
    x, y, y_sd=None,
    title="", ylab="", xlab="Date",
    smooth_window=None,
    show_legend=False,
    band_label="±1 SD",
    mean_label="Mean"
):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # Clean axes like your example
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=GRID_ALPHA, linewidth=0.8)

    # Band (± sd)
    if y_sd is not None:
        ax.fill_between(
            x, y - y_sd, y + y_sd,
            color=BLUE, alpha=BAND_ALPHA, linewidth=0,
            label=band_label
        )

    # Mean line
    # ax.plot(x, y, color=BLUE, linewidth=LINE_W, label=mean_label)

    # Smooth line (optional)
    if smooth_window is not None and smooth_window >= 3:
        y_s = _rolling_mean(y, smooth_window)
        ax.plot(x, y_s, color=BLUE, linewidth=2.6, alpha=0.95,
                label=f"Rolling mean ({smooth_window})")

    # Date formatting (nice)
    ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(x.min(), x.max())

    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)

    if show_legend:
        ax.legend(frameon=False, loc="upper left")

    plt.tight_layout()
    plt.show()


# ------------------------------
# 1) Bloom extent (km²) + band (proxy temporal)
# ------------------------------
x = df["date"]
y_area = df["bloom_area_km2"].astype(float).to_numpy()

# Para área bloom no tienes SD espacial; usamos banda proxy temporal para estética tipo paper
y_area_sd_proxy = rolling_std(y_area, SMOOTH_WINDOW)

plot_mean_with_band(
    x=x,
    y=y_area,
    y_sd=y_area_sd_proxy,
    title="Bloom extent (km²) — Huasco Basin (band = rolling SD proxy)",
    ylab="Bloom area (km²)",
    smooth_window=SMOOTH_WINDOW,
    show_legend=True,
    band_label="±1 SD (rolling proxy)",
    mean_label="Bloom area"
)


# ------------------------------
# 2) Bloom intensity vs basin NDVI mean
#    (líneas; y banda para basin si hay SD)
# ------------------------------
fig, ax = plt.subplots(figsize=(12.5, 3.6))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=GRID_ALPHA, linewidth=0.8)

y_bloom_mean = df[f"bloom_{variable}_mean"].astype(float).to_numpy()
y_basin_mean = df[f"basin_{variable}_mean"].astype(float).to_numpy()

# Banda SD espacial de la cuenca (si existe)
sd_col = f"basin_{variable}_sd"
if sd_col in df.columns:
    y_basin_sd = df[sd_col].astype(float).to_numpy()
    ax.fill_between(x, y_basin_mean - y_basin_sd, y_basin_mean + y_basin_sd,
                    color=BLUE, alpha=BAND_ALPHA, linewidth=0,
                    label=f"Basin {variable} ±1 SD")

# Líneas (promedios)
ax.plot(x, y_bloom_mean, color=BLUE, linewidth=LINE_W, label=f"Bloom {variable} mean")
ax.plot(x, y_basin_mean, color=BLUE, linewidth=LINE_W, alpha=0.55, label=f"Basin {variable} mean")

# Suavizados (opcional, para look tipo paper)
ax.plot(x, _rolling_mean(y_bloom_mean, SMOOTH_WINDOW), color=BLUE, linewidth=2.6, alpha=0.95,
        label=f"Bloom {variable} rolling mean ({SMOOTH_WINDOW})")
ax.plot(x, _rolling_mean(y_basin_mean, SMOOTH_WINDOW), color=BLUE, linewidth=2.6, alpha=0.45,
        label=f"Basin {variable} rolling mean ({SMOOTH_WINDOW})")

ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlim(x.min(), x.max())

ax.set_title(f"Bloom intensity vs Basin {variable} mean", fontsize=12, pad=10)
ax.set_xlabel("Date")
ax.set_ylabel(variable)
ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
plt.show()


# ------------------------------
# 3) Basin anomaly mean + band (real if basin_anom_sd exists; else proxy)
# ------------------------------
y_anom = df["basin_anom_mean"].astype(float).to_numpy()

if "basin_anom_sd" in df.columns:
    y_anom_sd = df["basin_anom_sd"].astype(float).to_numpy()
    band_label = "±1 SD (spatial)"
    title = "Basin anomaly mean ± SD — Huasco Basin"
else:
    y_anom_sd = _rolling_std(y_anom, SMOOTH_WINDOW)
    band_label = "±1 SD (rolling proxy)"
    title = "Basin anomaly mean — Huasco Basin (band = rolling SD proxy)"

plot_mean_with_band(
    x=x,
    y=y_anom,
    y_sd=y_anom_sd,
    title=title,
    ylab=f"{variable} anomaly",
    smooth_window=SMOOTH_WINDOW,
    show_legend=True,
    band_label=band_label,
    mean_label="Anomaly mean"
)


# ------------------------------
# (Optional extra) Scatter: bloom area vs bloom intensity
# ------------------------------
plt.figure(figsize=(6.4, 4.8))
plt.scatter(df["bloom_area_km2"], df[f"bloom_{variable}_mean"], alpha=0.6)
plt.title("Bloom area vs Bloom intensity")
plt.xlabel("Bloom area (km²)")
plt.ylabel(f"Bloom {variable} mean")
plt.grid(alpha=GRID_ALPHA)
plt.tight_layout()
plt.show()


## 8) Suggested GitHub repo layout

- `notebooks/huasco_bloom_mod13q1.ipynb`
- `data/huasco_bloom_timeseries_2001_2025.csv`
- `figures/` (optional exports from the plots)
- `README.md`

### Method summary (copy to README)
- Dataset: MODIS MOD13Q1 (16‑day NDVI), Collection 6.1  
- QA filter: SummaryQA ≤ 1  
- slot16: floor(DOY/16)  
- Climatology: mean NDVI per slot16  
- Anomaly: NDVI − climatology(slot16)  
- Bloom: NDVI > 0.20 AND Anomaly > 0.02  
- Metrics: bloom area (km²), bloom NDVI mean, basin NDVI mean/sd, basin anomaly mean


## 8) Top‑2 bloom periods and NDVI max composites

Here we:
1) Select the **two 16‑day periods** with the largest `bloom_area_km2`.
2) For each period, filter `modis_plus` to the **16‑day window** starting at that image date.
3) Compute a **max NDVI composite** (`NDVI_max`) from the images in that window.
4) Save quick‑look PNGs for your GitHub repo.


In [ ]:
# Select top-2 periods by bloom extent (km²)
top2 = (df.dropna(subset=['bloom_area_km2'])
          .sort_values('bloom_area_km2', ascending=False)
          .head(2)
          .reset_index(drop=True))

top2[['date','time_start','slot16','bloom_area_km2']]


In [ ]:
!pip -q install requests pillow

import os
import requests
from PIL import Image
from io import BytesIO

out_dir = 'figures'
os.makedirs(out_dir, exist_ok=True)

def ndvi_max_composite_for_period(time_start_millis, window_days=16):
    start = ee.Date(int(time_start_millis))
    end = start.advance(window_days, 'day')
    col = modis_plus.filterDate(start, end).select('NDVI')
    return col.max().rename('NDVI_max').set({
        'window_start': start.millis(),
        'window_end': end.millis()
    })

def save_thumbnail(img, region, vis, out_png, dimensions=1200):
    url = img.getThumbURL({
        'region': region,
        'dimensions': dimensions,
        'format': 'png',
        **vis
    })
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    im = Image.open(BytesIO(r.content))
    im.save(out_png)
    return out_png

ndvi_max_imgs = []
png_paths = []

vis_ndvi = {'min': 0, 'max': 0.8, 'palette': ndviVis_palette if 'ndviVis_palette' in globals() else [
    'ffffff','e8d5b7','d2b48c','b38b6d','d9f0a3','addd8e','78c679','41ab5d','238443','006837'
]}

# NOTE: In the notebook earlier we defined ndviVis as a dict in JS; here we keep a Python palette.
# If you want exactly the same palette as the JS script, paste it into `vis_ndvi['palette']`.

for i,row in top2.iterrows():
    t0 = row['time_start']
    d  = row['date']
    img = ndvi_max_composite_for_period(t0)
    ndvi_max_imgs.append(img)

    out_png = os.path.join(out_dir, f'ndvi_max_period_{i+1}_{pd.to_datetime(d).date()}.png')
    save_thumbnail(img.clip(REGION), REGION, vis_ndvi, out_png)
    png_paths.append(out_png)

png_paths

In [ ]:
# Display the two NDVI-max composites side-by-side
imgs = [Image.open(p) for p in png_paths]

plt.figure(figsize=(14, 6))
for i,im in enumerate(imgs):
    plt.subplot(1, 2, i+1)
    plt.imshow(im)
    plt.axis('off')
    plt.title(f"NDVI max — top bloom period {i+1}\n{top2.loc[i,'date']} (area={top2.loc[i,'bloom_area_km2']:.2f} km²)")
plt.tight_layout()
plt.show()


## 9) Event-based bloom analysis (smoothed peaks + inflection windows) and NDVI max composites

This section detects **bloom events** (multi-month episodes) from the **bloom extent time series**:

1. Smooth bloom area time series (Savitzky–Golay filter).
2. Detect **local maxima** (peaks) on the smoothed series.
3. For each peak, estimate event **start/end** using **inflection points** (sign changes of the 2nd derivative).
4. Select **Top-N events** by peak bloom area.
5. For each event window, compute a **NDVI max composite** from the MOD13Q1 images within that window and plot the results.

> Tune: `SG_WINDOW`, `PROMINENCE`, and `MIN_DISTANCE` to match the event scale you expect (e.g., 2002/2011/2017).


In [ ]:
# --- Dependencies for event detection ---
!pip -q install scipy

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks

# Safety checks
assert 'df' in globals(), "df not found. Run the section that builds the time series DataFrame first."
assert 'modis_plus' in globals(), "modis_plus not found. Run the Earth Engine processing section first."

# Ensure proper types and sorting
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Bloom area series (fill NA with 0 for event detection)
y = df['bloom_area_km2'].fillna(0).values.astype(float)

# --- Smoothing parameters (tune if needed) ---
SG_WINDOW = 15   # must be odd; ~15 points ~= 240 days for MOD13Q1 (16d steps)
SG_POLY   = 3

# Smooth bloom series
y_smooth = savgol_filter(y, SG_WINDOW, SG_POLY)
df['bloom_smooth'] = y_smooth

# --- Peak detection parameters (tune if needed) ---
MIN_DISTANCE = 10   # minimum separation between peaks (points) ~ 160 days
PROMINENCE   = 200  # km2 prominence; tune based on your basin scaling

peaks, props = find_peaks(
    y_smooth,
    distance=MIN_DISTANCE,
    prominence=PROMINENCE
)

print(f"Detected peaks: {len(peaks)}")
if len(peaks) > 0:
    display(df.loc[peaks, ['date', 'bloom_area_km2', 'bloom_smooth']].head(20))


In [ ]:
# --- Build event windows using inflection points (2nd derivative sign criteria) ---
dy  = np.gradient(y_smooth)
ddy = np.gradient(dy)

events = []
for p in peaks:
    # Start: walk left while curvature is still "rising" (ddy > 0), then stop
    s = p
    while s > 1 and ddy[s] > 0:
        s -= 1

    # End: walk right while curvature is still "falling" (ddy < 0), then stop
    e = p
    while e < len(ddy)-2 and ddy[e] < 0:
        e += 1

    events.append((s, p, e))

# Visualize series + detected event windows
plt.figure(figsize=(14,5))
plt.plot(df['date'], y, alpha=0.25, label='Bloom area (original)')
plt.plot(df['date'], y_smooth, linewidth=2, label='Bloom area (smoothed)')

for s, p, e in events:
    plt.axvspan(df['date'].iloc[s], df['date'].iloc[e], alpha=0.15)
    plt.scatter(df['date'].iloc[p], y_smooth[p], color='red')

plt.title("Bloom event detection: peaks + inflection-based windows")
plt.xlabel("Date")
plt.ylabel("Bloom area (km²)")
plt.legend()
plt.tight_layout()
plt.show()

# Build event table
event_rows = []
for s,p,e in events:
    event_rows.append({
        "start": df['date'].iloc[s],
        "peak":  df['date'].iloc[p],
        "end":   df['date'].iloc[e],
        "max_area_smooth_km2": float(y_smooth[p]),
        "max_area_raw_km2": float(y[p])
    })

events_df = pd.DataFrame(event_rows).sort_values("max_area_smooth_km2", ascending=False).reset_index(drop=True)
events_df.head(10)


In [ ]:
# --- Select Top-N bloom events ---
TOP_N = 2  # set to 3 if you want 2002, 2011, 2017, etc.

top_events = events_df.head(TOP_N).copy()
top_events


### NDVI max composite for each bloom event window

For each event `(start, end)`, we filter `modis_plus` to that window and compute:

- `NDVI_max = max(NDVI)` over all images in the event window.

Then we download rasters directly from Earth Engine to NumPy arrays (no Drive export) and plot side-by-side.


In [ ]:
import geemap
import ee
import numpy as np

# Helper: compute NDVI max composite for a given event window
def ndvi_max_composite(start, end):
    col = (modis_plus
           .filterDate(ee.Date(start), ee.Date(end))
           .select('NDVI'))
    return col.max().clip(REGION)  # REGION is basin geometry

# Helper: convert EE image to numpy array + extent
def ee_image_to_numpy_and_extent(img, region, scale=500):
    arr = geemap.ee_to_numpy(img, region=region, scale=scale)
    arr = np.squeeze(arr)
    # Compute extent from region bounds (lon/lat)
    bounds = ee.Geometry(region).bounds().coordinates().getInfo()[0]
    xs = [c[0] for c in bounds]
    ys = [c[1] for c in bounds]
    extent = [min(xs), max(xs), min(ys), max(ys)]
    return arr, extent

# Compute composites
composites = []
for _, row in top_events.iterrows():
    start = row['start']
    end   = row['end']
    peak  = row['peak']
    img = ndvi_max_composite(start, end)
    composites.append((start, peak, end, img))

print("✅ NDVI max composites prepared:", len(composites))


In [ ]:
# --- Plot NDVI max composites (Top-N events) ---
fig, axes = plt.subplots(1, len(composites), figsize=(6*len(composites), 6))

if len(composites) == 1:
    axes = [axes]

for ax, (start, peak, end, img) in zip(axes, composites):
    ndvi_np, extent = ee_image_to_numpy_and_extent(img, REGION, scale=SCALE_TS)
    im = ax.imshow(ndvi_np, extent=extent, origin='upper', vmin=0, vmax=0.8, cmap='YlGn')
    ax.set_title(f"NDVI max\n{start.date()} → {end.date()}\n(peak: {peak.date()})")
    ax.set_xlabel("Lon")
    ax.set_ylabel("Lat")

fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02, label="NDVI")
plt.suptitle(f"Top-{len(composites)} bloom events: NDVI max composites", fontsize=14, style='italic')
plt.tight_layout()
plt.show()


In [ ]:
# --- Save figures for GitHub repo (optional) ---
import os
os.makedirs("figures", exist_ok=True)

for i, (start, peak, end, img) in enumerate(composites, start=1):
    ndvi_np, extent = ee_image_to_numpy_and_extent(img, REGION, scale=SCALE_TS)
    plt.figure(figsize=(7,7))
    im = plt.imshow(ndvi_np, extent=extent, origin='upper', vmin=0, vmax=0.8, cmap='YlGn')
    plt.title(f"NDVI max composite\n{start.date()} → {end.date()} (peak: {peak.date()})")
    plt.xlabel("Lon"); plt.ylabel("Lat")
    plt.colorbar(im, fraction=0.046, pad=0.04, label="NDVI")
    out_png = f"figures/ndvi_max_event_{i}_{peak.date()}.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()
    print("✅ Saved:", out_png)
